In [ ]:
import sys
import os

sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

import modeling
import wifiplotting as wp
from wifiplotting import TL_CORNER, BR_CORNER

In [ ]:
sequoia_df = pd.read_csv("../../data/sequoia.csv", index_col="index").reset_index(drop=True)

In [ ]:
sequoia_df.head()

In [ ]:
osm_context = wp.OSMPlotContext.from_bounds(
    init_lons=[BR_CORNER[1], TL_CORNER[1]],
    init_lats=[BR_CORNER[0], TL_CORNER[0]],
)

plot_df = sequoia_df.dropna(subset=["latitude", "longitude", "rssi_sample", "indoor"]).copy()
plot_df["indoor_label"] = np.where(plot_df["indoor"].astype(bool), "Indoor", "Outdoor")

lat_bounds = (osm_context.bounds[1], osm_context.bounds[3])
lon_bounds = (osm_context.bounds[0], osm_context.bounds[2])

print(f"Using {len(plot_df):,} non-null RSSI observations from {len(sequoia_df):,} rows.")

# RSSI observations

In [ ]:
fig, ax, osm_metadata = osm_context.generate_base_axis(figsize=(12, 9))

x, y = osm_context.to_world(plot_df["longitude"].to_numpy(), plot_df["latitude"].to_numpy())
points = ax.scatter(
    x,
    y,
    c=plot_df["rssi_sample"],
    cmap="RdYlGn",
    s=18,
    alpha=0.75,
    edgecolors="none",
    zorder=3,
)

colorbar = fig.colorbar(points, ax=ax, shrink=0.88)
colorbar.set_label("RSSI sample")
ax.set_title("RSSI Observations")
plt.show()

# Binned mean RSSI

In [ ]:
def plot_binned_rssi_map(
    binned_df,
    *,
    value_col="mean",
    title,
    colorbar_label="Mean RSSI sample",
    cmap="RdYlGn",
):
    binned_df = binned_df.dropna(subset=[value_col])
    fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)
    norm = Normalize(vmin=binned_df[value_col].min(), vmax=binned_df[value_col].max())
    mapper = ScalarMappable(norm=norm, cmap=cmap)

    for ax, indoor_value, label in zip(axes, [True, False], ["Indoor", "Outdoor"]):
        osm_context.generate_base_axis(ax=ax)
        subset = binned_df[binned_df["indoor"].astype(bool) == indoor_value]

        for _, row in subset.iterrows():
            lat0, lat1 = row["lat_bounds"]
            lon0, lon1 = row["lon_bounds"]
            x0, y0 = osm_context.to_world(lon0, lat0)
            x1, y1 = osm_context.to_world(lon1, lat1)
            ax.add_patch(Rectangle(
                (min(x0, x1), min(y0, y1)),
                abs(x1 - x0),
                abs(y1 - y0),
                facecolor=mapper.to_rgba(row[value_col]),
                edgecolor="black",
                linewidth=0.35,
                alpha=0.78,
                zorder=3,
            ))

        ax.set_title(f"{title}: {label}")

    colorbar = fig.colorbar(mapper, ax=axes, shrink=0.82)
    colorbar.set_label(colorbar_label)
    return fig, axes

binned_rssi = modeling.binned_group_mean(
    plot_df,
    value="rssi_sample",
    lat_bins=30,
    lon_bins=30,
    lat_bounds=lat_bounds,
    lon_bounds=lon_bounds,
)
binned_rssi["indoor_label"] = np.where(binned_rssi["indoor"].astype(bool), "Indoor", "Outdoor")

plot_binned_rssi_map(binned_rssi, title="30 x 30 Binned Mean RSSI")
plt.show()

# Indoor vs outdoor RSSI distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), )

sns.histplot(
    data=plot_df,
    x="rssi_sample",
    hue="indoor_label",
    hue_order=["Indoor", "Outdoor"],
    bins=30,
    kde=True,
    alpha=0.3,
    ax=axes[0],
)
axes[0].set_title("RSSI Distribution (Unbinned)")
axes[0].set_xlabel("RSSI sample")

sns.histplot(
    data=binned_rssi,
    x="mean",
    hue="indoor_label",
    hue_order=["Indoor", "Outdoor"],
    bins=30,
    kde=True,
    alpha=0.3,
    ax=axes[1],
)
axes[1].set_title("RSSI Distribution (Binned Means)")
axes[1].set_xlabel("Mean RSSI sample per occupied bin")

plt.tight_layout()
plt.show()

# Mean RSSI by number of bins

In [ ]:
bin_counts = np.arange(5, 85, 5)
mean_by_bin_count = []

for n_bins in bin_counts:
    binned = modeling.binned_group_mean(
        plot_df,
        value="rssi_sample",
        lat_bins=n_bins,
        lon_bins=n_bins,
        lat_bounds=lat_bounds,
        lon_bounds=lon_bounds,
    )
    means = binned.groupby("indoor")["mean"].mean()

    for indoor_value, label in [(True, "Indoor"), (False, "Outdoor")]:
        mean_by_bin_count.append({
            "n_bins": n_bins,
            "environment": label,
            "mean_rssi": means.get(indoor_value, np.nan),
            "occupied_bins": int((binned["indoor"].astype(bool) == indoor_value).sum()),
        })

mean_by_bin_count = pd.DataFrame(mean_by_bin_count)

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(
    data=mean_by_bin_count,
    x="n_bins",
    y="mean_rssi",
    hue="environment",
    hue_order=["Indoor", "Outdoor"],
    marker="o",
    ax=ax,
)
ax.set_title("Indoor and Outdoor Mean RSSI over Grid Resolution")
ax.set_xlabel("Number of bins per latitude/longitude axis")
ax.set_ylabel("Mean of occupied-bin mean RSSI")
ax.grid(alpha=0.25)
plt.show()

mean_by_bin_count.head()

# RSSI variance over bin resolution

In [ ]:
raw_variance = (
    plot_df.groupby("indoor")["rssi_sample"]
    .var()
    .rename(index={True: "Indoor", False: "Outdoor"})
    .rename("raw_observation_variance")
    .reset_index()
    .rename(columns={"indoor": "environment"})
)

binned_mean_variance = []

for n_bins in bin_counts:
    binned = modeling.binned_group_mean(
        plot_df,
        value="rssi_sample",
        lat_bins=n_bins,
        lon_bins=n_bins,
        lat_bounds=lat_bounds,
        lon_bounds=lon_bounds,
    )
    variances = binned.groupby("indoor")["mean"].var()

    for indoor_value, label in [(True, "Indoor"), (False, "Outdoor")]:
        binned_mean_variance.append({
            "n_bins": n_bins,
            "environment": label,
            "variance_of_binned_means": variances.get(indoor_value, np.nan),
            "occupied_bins": int((binned["indoor"].astype(bool) == indoor_value).sum()),
        })

binned_mean_variance = pd.DataFrame(binned_mean_variance)

palette = {"Indoor": "tab:blue", "Outdoor": "tab:orange"}
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(
    data=binned_mean_variance,
    x="n_bins",
    y="variance_of_binned_means",
    hue="environment",
    hue_order=["Indoor", "Outdoor"],
    marker="o",
    palette=palette,
    ax=ax,
)

for _, row in raw_variance.iterrows():
    ax.axhline(
        row["raw_observation_variance"],
        color=palette[row["environment"]],
        linestyle="--",
        alpha=0.75,
        label=f"{row['environment']} raw observation variance",
    )

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Environment")
ax.set_title("RSSI Variance: Raw Observations vs Binned Means")
ax.set_xlabel("Number of bins per latitude/longitude axis")
ax.set_ylabel("RSSI variance")
ax.grid(alpha=0.25)
plt.show()

raw_variance, binned_mean_variance.head()